# Instagram Usage & Psychological Risk — Data Visualisation

Exploratory analysis combining both datasets (`instagram_usage_lifestyle.csv` + `instagram_users_lifestyle.csv`).
All plots are saved to `../results/data_visualisation/`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

# ── Light, presentation-ready style ─────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.edgecolor': '#CCCCCC',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.9,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': '#CCCCCC',
    'axes.spines.bottom': '#CCCCCC',
})

PALETTE  = sns.color_palette('Set2')
PALETTE2 = sns.color_palette('tab10')
TARGET   = 'perceived_stress_score'

RESULTS = Path('../results/data_visualisation')
RESULTS.mkdir(parents=True, exist_ok=True)
print(f'Saving plots to: {RESULTS.resolve()}')

## 1. Load & Combine Both Datasets

In [ ]:
print('Loading dataset 1 ...')
df1 = pd.read_csv('../dataset/instagram_usage_lifestyle.csv', low_memory=False)
print(f'  instagram_usage_lifestyle  : {df1.shape}')

print('Loading dataset 2 ...')
df2 = pd.read_csv('../dataset/instagram_users_lifestyle.csv', low_memory=False)
print(f'  instagram_users_lifestyle  : {df2.shape}')

# Both datasets have identical columns — concatenate and deduplicate on user_id
df_full = pd.concat([df1, df2], ignore_index=True)
if 'user_id' in df_full.columns:
    df_full = df_full.drop_duplicates(subset='user_id')
else:
    df_full = df_full.drop_duplicates()

print(f'\nCombined (deduplicated): {df_full.shape}')

# Sample 200 K for fast visualisation
SAMPLE = min(200_000, len(df_full))
df = df_full.sample(n=SAMPLE, random_state=42).reset_index(drop=True)
print(f'Working sample          : {df.shape}')

## 2. Dataset Overview

In [ ]:
print('=== Data Types ===')
print(df.dtypes.value_counts())
print('\n=== Descriptive Statistics (numeric) ===')
display(df.describe().T.round(2))

## 3. Missing Values

In [ ]:
missing_pct = df.isnull().mean().sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(14, 4))
if len(missing_pct) > 0:
    missing_pct.plot(kind='bar', ax=ax, color=PALETTE[0], edgecolor='white', width=0.7)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_xlabel('Feature')
    ax.set_ylabel('Missing Fraction')
    plt.xticks(rotation=45, ha='right')
else:
    ax.text(0.5, 0.5, 'No missing values in this sample',
            ha='center', va='center', fontsize=14, transform=ax.transAxes)

ax.set_title('Missing Values by Feature')
fig.tight_layout()
fig.savefig(RESULTS / 'missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Missing features: {len(missing_pct)}')

## 4. Target Variable Distribution

In [ ]:
target_data = df[TARGET].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Target Variable — Perceived Stress Score')

# Histogram + KDE
axes[0].hist(target_data, bins=60, color=PALETTE[0], edgecolor='white', alpha=0.85,
             density=True, label='Histogram')
target_data.plot.kde(ax=axes[0], color='#C0392B', lw=2.5, label='KDE')
axes[0].set_title('Distribution')
axes[0].set_xlabel('Perceived Stress Score')
axes[0].set_ylabel('Density')
axes[0].legend()

# Box plot
bp = axes[1].boxplot(target_data, patch_artist=True, widths=0.4,
                     medianprops=dict(color='#C0392B', lw=2))
bp['boxes'][0].set_facecolor(PALETTE[1])
bp['boxes'][0].set_alpha(0.8)
axes[1].set_title('Box Plot')
axes[1].set_ylabel('Perceived Stress Score')
axes[1].set_xticks([])

# Cumulative distribution
sorted_vals = np.sort(target_data)
cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
axes[2].plot(sorted_vals, cdf, color=PALETTE[2], lw=2)
axes[2].axvline(target_data.quantile(0.80), color='#E74C3C', lw=1.5,
                linestyle='--', label='80th pct (high stress threshold)')
axes[2].set_title('Cumulative Distribution')
axes[2].set_xlabel('Perceived Stress Score')
axes[2].set_ylabel('Cumulative Probability')
axes[2].legend()

fig.tight_layout()
fig.savefig(RESULTS / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean: {target_data.mean():.2f}  |  Median: {target_data.median():.2f}  |  Std: {target_data.std():.2f}')
print(f'80th percentile (high-stress threshold): {target_data.quantile(0.80):.2f}')

## 5. Instagram Usage Feature Distributions

In [ ]:
instagram_features = [
    'daily_active_minutes_instagram', 'sessions_per_day',
    'average_session_length_minutes', 'reels_watched_per_day',
    'stories_viewed_per_day', 'likes_given_per_day',
    'comments_written_per_day', 'posts_created_per_week',
    'dms_sent_per_week', 'dms_received_per_week',
    'ads_viewed_per_day', 'ads_clicked_per_day',
    'time_on_feed_per_day', 'time_on_reels_per_day',
    'time_on_explore_per_day', 'user_engagement_score',
]
instagram_features = [f for f in instagram_features if f in df.columns]

n_cols = 4
n_rows = int(np.ceil(len(instagram_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.2))
fig.suptitle('Instagram Usage Feature Distributions (n=200 K)', y=1.01)
axes = axes.flatten()

for i, feat in enumerate(instagram_features):
    data = df[feat].dropna()
    axes[i].hist(data, bins=50, color=PALETTE2[i % len(PALETTE2)],
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=10)
    axes[i].set_ylabel('Count')
    # Annotate median
    med = data.median()
    axes[i].axvline(med, color='#C0392B', lw=1.5, linestyle='--',
                    label=f'Median={med:.1f}')
    axes[i].legend(fontsize=8)

for j in range(len(instagram_features), len(axes)):
    axes[j].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS / 'instagram_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plotted {len(instagram_features)} Instagram usage features')

## 6. Lifestyle & Health Feature Distributions

In [ ]:
lifestyle_features = [
    'exercise_hours_per_week', 'sleep_hours_per_night', 'body_mass_index',
    'daily_steps_count', 'weekly_work_hours', 'blood_pressure_systolic',
    'blood_pressure_diastolic', 'hobbies_count', 'social_events_per_month',
    'volunteer_hours_per_month', 'books_read_per_year', 'self_reported_happiness',
]
lifestyle_features = [f for f in lifestyle_features if f in df.columns]

n_cols = 4
n_rows = int(np.ceil(len(lifestyle_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.2))
fig.suptitle('Lifestyle & Health Feature Distributions', y=1.01)
axes = axes.flatten()

for i, feat in enumerate(lifestyle_features):
    data = df[feat].dropna()
    axes[i].hist(data, bins=50, color=PALETTE[i % len(PALETTE)],
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=10)
    axes[i].set_ylabel('Count')
    med = data.median()
    axes[i].axvline(med, color='#C0392B', lw=1.5, linestyle='--',
                    label=f'Median={med:.1f}')
    axes[i].legend(fontsize=8)

for j in range(len(lifestyle_features), len(axes)):
    axes[j].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS / 'lifestyle_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Demographic Distributions

In [ ]:
cat_features = ['gender', 'income_level', 'employment_status',
                'education_level', 'urban_rural', 'relationship_status']
cat_features = [f for f in cat_features if f in df.columns]

n_cols = 3
n_rows = int(np.ceil(len(cat_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
fig.suptitle('Demographic Feature Distributions', y=1.01)
axes = axes.flatten()

for i, feat in enumerate(cat_features):
    counts = df[feat].value_counts().sort_values(ascending=False)
    bars = axes[i].bar(range(len(counts)), counts.values,
                       color=[PALETTE2[j % len(PALETTE2)] for j in range(len(counts))],
                       edgecolor='white', width=0.7)
    axes[i].set_xticks(range(len(counts)))
    axes[i].set_xticklabels(counts.index, rotation=30, ha='right', fontsize=9)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=11)
    axes[i].set_ylabel('Count')
    # Annotate counts on bars
    for bar, val in zip(bars, counts.values):
        axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                     f'{val:,}', ha='center', va='bottom', fontsize=7)

# Age histogram
if 'age' in df.columns and i + 1 < len(axes):
    idx = i + 1
    axes[idx].hist(df['age'].dropna(), bins=30, color=PALETTE[3],
                   edgecolor='white', alpha=0.85)
    axes[idx].set_title('Age Distribution', fontsize=11)
    axes[idx].set_xlabel('Age')
    axes[idx].set_ylabel('Count')
    idx += 1
    for j in range(idx, len(axes)):
        axes[j].set_visible(False)
else:
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS / 'demographic_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Correlation Heatmaps

In [ ]:
# Select numeric columns; drop pure IDs
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude  = ['user_id', 'account_creation_year', 'linked_accounts_count']
num_cols = [c for c in num_cols if c not in exclude]

cmap = sns.diverging_palette(220, 20, as_cmap=True)

for method, label in [('pearson', 'Pearson'), ('spearman', 'Spearman')]:
    corr = df[num_cols].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))

    fig, ax = plt.subplots(figsize=(18, 15))
    sns.heatmap(corr, mask=mask, cmap=cmap, center=0,
                square=True, linewidths=0.4, cbar_kws={'shrink': 0.75},
                annot=False, ax=ax)
    ax.set_title(f'{label} Correlation Matrix — All Numeric Features', pad=20)
    fig.tight_layout()
    fig.savefig(RESULTS / f'{method}_correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {method}_correlation_heatmap.png')

## 9. Top Features vs Stress Score

In [ ]:
corr_with_target = (
    df[num_cols].corrwith(df[TARGET])
    .abs()
    .drop(TARGET, errors='ignore')
    .sort_values(ascending=False)
)
top8 = corr_with_target.head(8).index.tolist()

print('Top 8 features by |Pearson r| with stress:')
print(corr_with_target.head(8).round(4).to_string())

rng = np.random.default_rng(42)
idx = rng.choice(len(df), size=min(6000, len(df)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Top 8 Features vs Perceived Stress Score')
axes = axes.flatten()

for i, feat in enumerate(top8):
    x = df[feat].iloc[idx].values
    y = df[TARGET].iloc[idx].values
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]

    axes[i].scatter(x, y, alpha=0.15, s=8, color=PALETTE2[i % len(PALETTE2)])

    if len(x) > 2:
        m, b, r, p, _ = stats.linregress(x, y)
        xs = np.linspace(x.min(), x.max(), 200)
        axes[i].plot(xs, m * xs + b, color='#C0392B', lw=2,
                     label=f'r = {r:.3f}')
        axes[i].legend(fontsize=9)

    axes[i].set_xlabel(feat.replace('_', ' ').title(), fontsize=9)
    axes[i].set_ylabel('Stress Score', fontsize=9)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=10)

fig.tight_layout()
fig.savefig(RESULTS / 'top_features_vs_stress.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Stress Score by Demographic Group

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Mean Perceived Stress Score by Demographic Group')

# Age groups
if 'age' in df.columns:
    df['_age_group'] = pd.cut(df['age'],
                              bins=[0, 20, 30, 40, 50, 65, 120],
                              labels=['<20', '20-29', '30-39', '40-49', '50-64', '65+'])
    ag = df.groupby('_age_group', observed=True)[TARGET].agg(['mean', 'sem']).reset_index()
    axes[0, 0].bar(ag['_age_group'].astype(str), ag['mean'],
                   yerr=ag['sem'] * 1.96, capsize=4,
                   color=PALETTE[:len(ag)], edgecolor='white', width=0.65)
    axes[0, 0].set_title('Stress by Age Group')
    axes[0, 0].set_xlabel('Age Group')
    axes[0, 0].set_ylabel('Mean Stress Score')

# Gender
if 'gender' in df.columns:
    gd = df.groupby('gender')[TARGET].agg(['mean', 'sem']).sort_values('mean', ascending=False).reset_index()
    axes[0, 1].bar(gd['gender'].astype(str), gd['mean'],
                   yerr=gd['sem'] * 1.96, capsize=4,
                   color=PALETTE2[:len(gd)], edgecolor='white', width=0.65)
    axes[0, 1].set_title('Stress by Gender')
    axes[0, 1].set_xlabel('Gender')
    axes[0, 1].set_ylabel('Mean Stress Score')
    plt.setp(axes[0, 1].get_xticklabels(), rotation=20, ha='right')

# Income
if 'income_level' in df.columns:
    inc_order = ['Low', 'Medium', 'High']
    ic = df.groupby('income_level')[TARGET].agg(['mean', 'sem']).reset_index()
    ic['income_level'] = pd.Categorical(ic['income_level'], categories=inc_order, ordered=True)
    ic = ic.sort_values('income_level')
    axes[1, 0].bar(ic['income_level'].astype(str), ic['mean'],
                   yerr=ic['sem'] * 1.96, capsize=4,
                   color=PALETTE[:len(ic)], edgecolor='white', width=0.65)
    axes[1, 0].set_title('Stress by Income Level')
    axes[1, 0].set_xlabel('Income Level')
    axes[1, 0].set_ylabel('Mean Stress Score')

# Employment
if 'employment_status' in df.columns:
    em = df.groupby('employment_status')[TARGET].agg(['mean', 'sem']).sort_values('mean', ascending=True).reset_index()
    axes[1, 1].barh(em['employment_status'].astype(str), em['mean'],
                    xerr=em['sem'] * 1.96, capsize=4,
                    color=PALETTE2[:len(em)], edgecolor='white', height=0.65)
    axes[1, 1].set_title('Stress by Employment Status')
    axes[1, 1].set_xlabel('Mean Stress Score')

fig.tight_layout()
fig.savefig(RESULTS / 'stress_by_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Nonlinearity / Threshold Analysis

In [ ]:
threshold_feats = [
    'daily_active_minutes_instagram', 'likes_given_per_day',
    'time_on_reels_per_day', 'sessions_per_day',
    'stories_viewed_per_day', 'user_engagement_score',
]
threshold_feats = [f for f in threshold_feats if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Nonlinear Threshold Analysis: Mean Stress by Usage Decile')
axes = axes.flatten()

for i, feat in enumerate(threshold_feats):
    tmp = df[[feat, TARGET]].dropna().copy()
    tmp['decile'] = pd.qcut(tmp[feat], q=10, labels=False, duplicates='drop')
    grp = tmp.groupby('decile')[TARGET].agg(['mean', 'std', 'count']).reset_index()
    grp['sem'] = grp['std'] / np.sqrt(grp['count'])

    ax = axes[i]
    ax.plot(grp['decile'], grp['mean'], marker='o', color=PALETTE[i % len(PALETTE)],
            lw=2.2, markersize=7, label='Mean stress')
    ax.fill_between(grp['decile'],
                    grp['mean'] - 1.96 * grp['sem'],
                    grp['mean'] + 1.96 * grp['sem'],
                    alpha=0.25, color=PALETTE[i % len(PALETTE)], label='95% CI')
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('Usage Decile  (0 = lowest, 9 = highest)')
    ax.set_ylabel('Mean Stress Score')
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(RESULTS / 'nonlinearity_decile_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Happiness vs Stress (coloured by Instagram usage)

In [ ]:
if 'self_reported_happiness' in df.columns:
    rng2 = np.random.default_rng(0)
    idx2 = rng2.choice(len(df), size=min(15_000, len(df)), replace=False)
    sub  = df.iloc[idx2].dropna(subset=['self_reported_happiness', TARGET])

    colour_col = 'daily_active_minutes_instagram'
    fig, ax = plt.subplots(figsize=(11, 7))

    sc = ax.scatter(sub['self_reported_happiness'], sub[TARGET],
                    c=sub[colour_col] if colour_col in sub.columns else PALETTE[0],
                    cmap='YlOrRd', alpha=0.35, s=12, edgecolors='none')

    if colour_col in sub.columns:
        cbar = fig.colorbar(sc, ax=ax, shrink=0.85)
        cbar.set_label('Daily Active Minutes (Instagram)', fontsize=10)

    ax.set_xlabel('Self-Reported Happiness')
    ax.set_ylabel('Perceived Stress Score')
    ax.set_title('Happiness vs Stress Score (coloured by Instagram usage)')

    fig.tight_layout()
    fig.savefig(RESULTS / 'happiness_vs_stress.png', dpi=150, bbox_inches='tight')
    plt.show()

## 13. Violin Plots — Stress by Key Categorical Features

In [ ]:
violin_cats = ['gender', 'income_level', 'urban_rural']
violin_cats = [c for c in violin_cats if c in df.columns]

fig, axes = plt.subplots(1, len(violin_cats), figsize=(7 * len(violin_cats), 6))
fig.suptitle('Stress Score Distribution by Categorical Features')

if len(violin_cats) == 1:
    axes = [axes]

for i, cat in enumerate(violin_cats):
    order = df[cat].value_counts().index.tolist()
    sns.violinplot(data=df, x=cat, y=TARGET, order=order,
                   palette='Set2', inner='quartile', ax=axes[i])
    axes[i].set_title(cat.replace('_', ' ').title())
    axes[i].set_xlabel(cat.replace('_', ' ').title())
    axes[i].set_ylabel('Perceived Stress Score')
    plt.setp(axes[i].get_xticklabels(), rotation=25, ha='right')

fig.tight_layout()
fig.savefig(RESULTS / 'violin_stress_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Summary Statistics Table

In [ ]:
key_cols = [
    TARGET, 'self_reported_happiness',
    'daily_active_minutes_instagram', 'sessions_per_day',
    'likes_given_per_day', 'time_on_reels_per_day',
    'exercise_hours_per_week', 'sleep_hours_per_night', 'age',
]
key_cols = [c for c in key_cols if c in df.columns]

summary = df[key_cols].describe().T.round(2)
summary.columns = ['N', 'Mean', 'Std', 'Min', 'Q25', 'Median', 'Q75', 'Max']
summary['N'] = summary['N'].astype(int)
print('=== Key Feature Summary Statistics ===')
display(summary)

# Save as CSV for the report
summary.to_csv(RESULTS / 'summary_statistics.csv')
print('\nAll data visualisation plots saved to:', RESULTS.resolve())